In [183]:
import numpy as np
import re

In [260]:
# Routines

def figures_routine(lines):
    latex = "".join(lines)
    ruta = re.search(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]*)\}', latex)
    caption = re.search(r'\\caption\{(.*?)\}', latex, re.DOTALL)
    label = re.search(r'\\label\{([^}]*)\}', latex)
    if ruta and caption and label:
        xml = [f"""<figura id="{label.group(1)}">\n""",
        f"""    <ruta>{ruta.group(1)}</ruta>\n""",
        f"""    <caption>\n""",
        f"""        {caption.group(1)}\n""",
        f"""    </caption>\n""",
        f"""</figura>\n"""]
    return xml

def eqs_routine(lines):
    latex = "".join(lines)
    label = re.search(r'\\label\{([^}]*)\}', latex)
    eq = latex.split("\\be")[1].split("\\ee")[0].replace("\n","").split("\\label")[0]
    #eq = latex.split("\n")[1].split("\\label")[0]
    if label:
        xml=[f"""<ecuaciÃ³n id="{label.group(1)}">\n""",
             f"""    <![CDATA[ {eq} ]]>\n""", 
             """</ecuaciÃ³n>\n"""]
    else:
        xml=["""<ecuaciÃ³n>\n""",
             f"""    <![CDATA[ {eq} ]]>\n""", 
             """</ecuaciÃ³n>\n"""]
    return xml

def align_routine(lines):
    latex = "".join(lines)
    latex1 = latex.split("\\begin{alig")[1].split("\\end{align}")[0]
    label =  re.findall(r'\\label\{([^}]*)\}', latex1)
    eqs=latex.split("\\begin{align}")[1].split("\\end{align}")[0].replace("\n","").replace("&","").split("\\\\")
    if label:
        xml=["""<ecuaciÃ³n>\n"""]
        for i, eq_ in enumerate(eqs):
            eq = eq_.split("\\label")[0].replace(""""   ""","")
            if label[i]: 
                xml.extend([f"""    <lÃ\xadnea id="{label[i]}">""",f"""<![CDATA[ {eq} ]]>""","""</lÃ\xadnea>\n"""])
            else: xml.extend(["""    <lÃ\xadnea>""",f"""<![CDATA[ {eq} ]]>""","""</lÃ\xadnea>\n"""])
        xml.append("""</ecuaciÃ³n>\n""")
    else:
        xml=[f"""<ecuaciÃ³n id="{label}">\n""",
             f"""    <lÃ\xadnea>"""] + eqs + ["""</lÃ\xadnea>\n""",
             """</ecuaciÃ³n>\n"""]
    return xml

In [294]:
with open('ch1.txt','r') as f:
    lines = f.readlines()

In [295]:
xml_lines=[]; i=0; enable_paragraph = False

while i<len(lines):
    line = lines[i]
    if ("<" in line) or (">" in line): print(f"WARNING of '<' or '>' in line {i}")
    if "table" in line: print(f"Notification of \\table in line {i}")
    if "&" in line: print(f"WARNING of '&' in line {i}")

    if "\\begin{figure}" in line:
        for j, jline in enumerate(lines[i:]):
            if "\\end{figure}" in jline:
                xml = figures_routine(lines[i:i+j+1])
                xml_lines.extend(xml)
                i += j
                break
    elif "\\begin{subequations}" in line:
        for j, jline in enumerate(lines[i:]):
            if "\\end{subequations}" in jline:
                xml = align_routine(lines[i+1:i+j])
                xml_lines.extend(xml)
                i += j
                break
    elif ("\\be\n" in line) or ("\\be \n" in line):
        for j, jline in enumerate(lines[i:]):
            if "\\ee" in jline:
                xml = eqs_routine(lines[i:i+j+1])
                xml_lines.extend(xml)
                i += j
                break
    elif line == "\n":
        pass
    else:
        if enable_paragraph:
            xml=["<pÃ¡rrafo>",""" """+line, "</pÃ¡rrafo>\n"]
            xml_lines.extend(xml)
        else: xml_lines.append(line)
    i+=1 

WARNING of '<' or '>' in line 14
WARNING of '<' or '>' in line 16
WARNING of '<' or '>' in line 18


In [296]:
#Add extra tabs
num = 2
for i in range(len(xml_lines)):
    xml_lines[i]="\t"*num + xml_lines[i]

In [297]:
with open('output.txt', 'w') as f:
    f.writelines(xml_lines)